# step-1:Import required softwares

In [3]:
!pip install -U langchain_community langchain-huggingface
!pip install faiss-cpu
!pip install openai
!pip install google-genai
!pip install groq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

import faiss
import os

from langchain_community.vectorstores import FAISS



   ---------------------------------------- 0.0/574.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/574.7 kB ? eta -:--:--
   ------------------ --------------------- 262.1/574.7 kB ? eta -:--:--
   ---------------------------------------- 574.7/574.7 kB 1.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/883.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/883.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/883.6 kB ? eta -:--:--
   ----------------------- ---------------- 524.3/883.6 kB 1.7 MB/s eta 0:00:01
   ----------------------------------- ---- 786.4/883.6 kB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 883.6/883.6 kB 1.0 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.11.1
    Uninstalling langsmith-0.11.1:
      Successfully uninstalled langsmith-0.11.1
  Attempting uninstall: langchain-core
    Found existing installation: langchain-c


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\saiam\AppData\Local\Temp\ipykernel_6188\2637033946.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# step-2: load tcs pdf file and load all pages

In [2]:
loader=PyPDFLoader('tcs.pdf')
mydocuments=loader.load()

In [3]:
mydocuments[0]

Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.3 (Windows)', 'creationdate': '2026-05-15T23:33:43+05:30', 'moddate': '2026-08-12T17:07:19+05:30', 'title': 'TCS Annual Report 2025-26', 'trapped': '/false', 'source': 'tcs.pdf', 'total_pages': 360, 'page': 0, 'page_label': '1'}, page_content='Integrated\n/A.caltnnual Report\n2025-26\nLeadership in \nEnterprise /A.caltI\nInfrastructure \nto Intelligence')

In [4]:
mydocuments[0].page_content

'Integrated\n/A.caltnnual Report\n2025-26\nLeadership in \nEnterprise /A.caltI\nInfrastructure \nto Intelligence'

In [5]:
len(mydocuments[0].page_content)

105

# step-3: convert documents to chunks

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2500,
			chunk_overlap=500)
mychunks = text_splitter.split_documents(mydocuments)

# step 4 - take vector model

In [7]:
model = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-mpnet-base-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

# step 5 - create faiss db and pass documents and vector model

In [8]:
vector_store = FAISS.from_documents(mychunks, model)

# Step 6 - Ask a questions on tcs pdf get relevant chunks from faiss db

In [9]:
results = vector_store.similarity_search("how is the tcs performing in terms of revenue?",k=3)

In [10]:
print(type(results))
print(type(results[0]))
print(results[0])


<class 'list'>
<class 'langchain_core.documents.base.Document'>
page_content='of margins.
These were offset by investments in talent and capability building, 
and other investments for strengthening ecosystem partnerships and 
increasing Go-to-Market pipeline.
Operating Profit Trend (H crore)
FY  2022
48, 453
FY  2024
59 , 31 1
FY  2023
54, 2 37
FY  2025
62 , 1 65
FY  2026
66, 838
2 5. 3% 2 5. 0 %
2 4. 3%2 4. 6%2 4. 1 %
O perati ng  P rof i t
O perati ng  Mar g i n
Over the past five fiscal years, TCS has steadily increased its Earnings 
Per Share (EPS), recording a compound annual growth rate of 10.3% . 
This consistent upward trend demonstrates not only the company’s 
strengthening profits but also its dedication to providing sustained value 
for shareholders.
Earnings per share (Amount in H)
10. 3%
C AG R
FY  2022
 1 0 3. 62  
FY  2024
 1 2 7. 74 
FY  2023
 1 1 5. 1 9  
FY  2025
 1 34. 1 9  
FY  2026
 1 45. 9 9  
TCS has demonstrated a strong cash conversion ratio, which reflects it

# Step 7 - Keep all chunks into one variable

In [11]:
for i, res in enumerate(results):
    print(i,type(res))
    

0 <class 'langchain_core.documents.base.Document'>
1 <class 'langchain_core.documents.base.Document'>
2 <class 'langchain_core.documents.base.Document'>


In [12]:
our_context = ''
for res in results:
    if isinstance(res,str):
        our_context += res+ '\n\n'
    else:
        our_context += res.page_content + '\n\n'
print(our_context)

of margins.
These were offset by investments in talent and capability building, 
and other investments for strengthening ecosystem partnerships and 
increasing Go-to-Market pipeline.
Operating Profit Trend (H crore)
FY  2022
48, 453
FY  2024
59 , 31 1
FY  2023
54, 2 37
FY  2025
62 , 1 65
FY  2026
66, 838
2 5. 3% 2 5. 0 %
2 4. 3%2 4. 6%2 4. 1 %
O perati ng  P rof i t
O perati ng  Mar g i n
Over the past five fiscal years, TCS has steadily increased its Earnings 
Per Share (EPS), recording a compound annual growth rate of 10.3% . 
This consistent upward trend demonstrates not only the company’s 
strengthening profits but also its dedication to providing sustained value 
for shareholders.
Earnings per share (Amount in H)
10. 3%
C AG R
FY  2022
 1 0 3. 62  
FY  2024
 1 2 7. 74 
FY  2023
 1 1 5. 1 9  
FY  2025
 1 34. 1 9  
FY  2026
 1 45. 9 9  
TCS has demonstrated a strong cash conversion ratio, which reflects its solid 
financial position and effective generation of operational cash. 
A r

# Step 8 - Make a prompt

In [13]:
question = input('enter your question')
our_prompt = f"""
I am going to ask a question based on my context.
please answer the question based on my context only.
if you don't find the answer in my context, then tell me you didn't find the answer.
don't give me an answer from your knowledge.

context={our_context}

question = {question}
"""

print(our_prompt)

enter your question how tcs is performing interms of revenue



I am going to ask a question based on my context.
please answer the question based on my context only.
if you don't find the answer in my context, then tell me you didn't find the answer.
don't give me an answer from your knowledge.

context=of margins.
These were offset by investments in talent and capability building, 
and other investments for strengthening ecosystem partnerships and 
increasing Go-to-Market pipeline.
Operating Profit Trend (H crore)
FY  2022
48, 453
FY  2024
59 , 31 1
FY  2023
54, 2 37
FY  2025
62 , 1 65
FY  2026
66, 838
2 5. 3% 2 5. 0 %
2 4. 3%2 4. 6%2 4. 1 %
O perati ng  P rof i t
O perati ng  Mar g i n
Over the past five fiscal years, TCS has steadily increased its Earnings 
Per Share (EPS), recording a compound annual growth rate of 10.3% . 
This consistent upward trend demonstrates not only the company’s 
strengthening profits but also its dedication to providing sustained value 
for shareholders.
Earnings per share (Amount in H)
10. 3%
C AG R
FY  2022
 1 0 3

# step 9 - Give prompt to open AI LLM And get response

In [ ]:

from groq import Groq
import os

# Enter your actual Groq API key
os.environ["GROQ_API_KEY"] = "Groq_API_KEY"

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

our_prompt = "What is tcs revenue?"

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": our_prompt}
    ]
)

print(response.choices[0].message.content)


**Tata Consultancy Services (TCS) – Revenue Snapshot (as of the most recent publicly‑available data)**  

| Fiscal Year (ended 31 Mar) | Revenue (INR) | Approx. Revenue (USD\*) | YoY Growth |
|----------------------------|--------------|------------------------|------------|
| FY 2023‑24 (2024)          | **₹ 2.86 trillion** | **≈ $34 – $35 billion** | + 16 % |
| FY 2022‑23 (2023)          | ₹ 2.64 trillion | ≈ $31 billion | + 13 % |
| FY 2021‑22 (2022)          | ₹ 2.24 trillion | ≈ $26 billion | + 13 % |

\* USD conversion uses an average exchange‑rate for the respective year (≈ ₹ 83 / $1 for FY 2023‑24; ≈ ₹ 84 / $1 for FY 2022‑23). Exact dollar amounts will vary with the spot rate you apply.

### Key points

1. **Largest Indian IT services firm** – TCS is consistently the biggest revenue generator among Indian IT services companies and one of the largest globally.  
2. **Revenue composition** – The bulk of the revenue comes from **digital & consulting services, application developme